###  Cell 1 — Setup

In [0]:
# ============================================================
#  04_GOLD_AGGREGATION PART 2 — TOURGO DATA PIPELINE
#  Gold Table 4: gold_booking_funnel
#  Gold Table 5: gold_payment_analysis
#  Gold Table 6: gold_review_summary
# ============================================================

from pyspark.sql.functions import (
    col, count, avg, sum as _sum,
    round as _round, when, lit
)

spark.sql("USE tourgo")

# Đọc Silver tables
df_bookings = spark.read.table("silver_bookings")
df_payments  = spark.read.table("silver_payments")
df_reviews   = spark.read.table("silver_reviews")
df_tours     = spark.read.table("silver_tours")

print("    Đọc Silver tables xong")
print(f"   bookings : {df_bookings.count():,}")
print(f"   payments : {df_payments.count():,}")
print(f"   reviews  : {df_reviews.count():,}")
print(f"   tours    : {df_tours.count():,}")

    Đọc Silver tables xong
   bookings : 1,326
   payments : 794
   reviews  : 272
   tours    : 779


### Cell 2 — Gold Table 4: Booking Funnel

In [0]:
# ── GOLD TABLE 4: Booking Funnel ───────────────────────────
print("\n Đang tạo gold_booking_funnel...")

total_bookings = df_bookings.count()

df_gold_funnel = df_bookings \
    .groupBy("status") \
    .agg(count("id").alias("count")) \
    .withColumn(
        "percentage",
        _round(col("count") / lit(total_bookings) * 100, 2)
    ) \
    .orderBy("status")

df_gold_funnel.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_booking_funnel")

print(f"gold_booking_funnel: {df_gold_funnel.count()} status rows")
spark.read.table("gold_booking_funnel").show(truncate=False)


 Đang tạo gold_booking_funnel...
gold_booking_funnel: 3 status rows
+---------+-----+----------+
|status   |count|percentage|
+---------+-----+----------+
|cancelled|276  |20.81     |
|confirmed|797  |60.11     |
|pending  |253  |19.08     |
+---------+-----+----------+



### Cell 3 — Gold Table 5: Payment Analysis

In [0]:
# ── GOLD TABLE 5: Payment Analysis ─────────────────────────
print("\nĐang tạo gold_payment_analysis...")

# Phân tích theo payment_method + status
df_payment_method = df_payments \
    .groupBy("payment_method", "status") \
    .agg(
        count("id").alias("count"),
        _round(_sum("amount"), 0).alias("total_amount")
    ) \
    .orderBy("payment_method", "status")

# Tính success rate theo method
df_payment_success = df_payments \
    .groupBy("payment_method") \
    .agg(
        count("id").alias("total_attempts"),
        count(when(col("status") == "SUCCESS", 1)).alias("successful")
    ) \
    .withColumn(
        "success_rate",
        _round(col("successful") / col("total_attempts") * 100, 2)
    )

# Lưu bảng method + status (bảng chính)
df_payment_method.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_payment_analysis")

# Lưu success rate (bảng phụ để dùng trong dashboard)
df_payment_success.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_payment_success_rate")

print(f" gold_payment_analysis: {df_payment_method.count()} rows")
print(f" gold_payment_success_rate: {df_payment_success.count()} rows")

print("\n Payment Method Breakdown:")
spark.read.table("gold_payment_analysis").show(truncate=False)

print("\n Success Rate by Method:")
spark.read.table("gold_payment_success_rate").show(truncate=False)


Đang tạo gold_payment_analysis...
 gold_payment_analysis: 5 rows
 gold_payment_success_rate: 4 rows

 Payment Method Breakdown:
+--------------+--------------+-----+------------+
|payment_method|status        |count|total_amount|
+--------------+--------------+-----+------------+
|BANK_TRANSFER |SUCCESS       |241  |1.92065E9   |
|CASH          |SUCCESS       |274  |2.50255E9   |
|VNPAY         |SUCCESS       |269  |1.95905E9   |
|VietQR        |AWAITING_ADMIN|6    |5332222.0   |
|VietQR        |SUCCESS       |4    |5140000.0   |
+--------------+--------------+-----+------------+


 Success Rate by Method:
+--------------+--------------+----------+------------+
|payment_method|total_attempts|successful|success_rate|
+--------------+--------------+----------+------------+
|VNPAY         |269           |269       |100.0       |
|BANK_TRANSFER |241           |241       |100.0       |
|VietQR        |10            |4         |40.0        |
|CASH          |274           |274       |100.0  

### Cell 4 — Gold Table 6: Review Summary

In [0]:
# ── GOLD TABLE 6: Review Summary ───────────────────────────
print("\n  Đang tạo gold_review_summary...")

# Join reviews -> tours để lấy title, category
df_review_summary = df_reviews \
    .join(
        df_tours.select(
            col("id").alias("t_id"),
            col("title"),
            col("category_names")
        ),
        col("tour_id") == col("t_id"),
        "left"
    ) \
    .drop("t_id") \
    .groupBy("tour_id", "title", "category_names") \
    .agg(
        _round(avg("rating"), 2).alias("avg_rating"),
        count("id").alias("review_count"),
        count(when(col("rating") == 5, 1)).alias("five_star"),
        count(when(col("rating") == 4, 1)).alias("four_star"),
        count(when(col("rating") == 1, 1)).alias("one_star")
    ) \
    .orderBy(col("avg_rating").desc())

# Rating distribution tổng thể (dùng cho bar chart)
df_rating_dist = df_reviews \
    .groupBy("rating") \
    .agg(count("id").alias("count")) \
    .orderBy("rating")

df_review_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_review_summary")

df_rating_dist.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_rating_distribution")

print(f" gold_review_summary: {df_review_summary.count()} tours")
print(f" gold_rating_distribution: {df_rating_dist.count()} rating levels")

print("\n Top Rated Tours:")
spark.read.table("gold_review_summary").show(10, truncate=False)

print("\n Rating Distribution:")
spark.read.table("gold_rating_distribution").show(truncate=False)


  Đang tạo gold_review_summary...
 gold_review_summary: 198 tours
 gold_rating_distribution: 3 rating levels

 Top Rated Tours:
+-------+-----------------------------+-----------------------+----------+------------+---------+---------+--------+
|tour_id|title                        |category_names         |avg_rating|review_count|five_star|four_star|one_star|
+-------+-----------------------------+-----------------------+----------+------------+---------+---------+--------+
|189    |Trải nghiệm Cần Thơ cuối tuần|Sang trọng,Mạo hiểm    |5.0       |1           |1        |0        |0       |
|562    |Tour cao cấp Tây Ninh 4N3Đ   |Sang trọng             |5.0       |1           |1        |0        |0       |
|200    |Tour cao cấp Quy Nhơn 2N1Đ   |Văn hóa                |5.0       |1           |1        |0        |0       |
|256    |Tham quan Vũng Tàu 5N4Đ      |Núi,Nghỉ dưỡng         |5.0       |1           |1        |0        |0       |
|230    |Tour cao cấp Mũi Né 5N4Đ     |Núi,Văn hóa  

### Cell 5 — Gold Summary Report (chụp screenshot)

In [0]:
# ============================================================
#  GOLD LAYER COMPLETE SUMMARY — chụp screenshot toàn bộ
# ============================================================

ALL_GOLD = [
    "gold_revenue_daily",        # Day 3
    "gold_provider_performance", # Day 3
    "gold_tour_performance",     # Day 3
    "gold_booking_funnel",       # Day 4
    "gold_payment_analysis",     # Day 4
    "gold_payment_success_rate", # Day 4
    "gold_review_summary",       # Day 4
    "gold_rating_distribution",  # Day 4
]

print("\n" + "="*55)
print(" GOLD LAYER COMPLETE — DAY 3 + DAY 4")
print("="*55)
print(f"{'Table':<30} | {'Records':>8} | Status")
print("-"*50)

for t in ALL_GOLD:
    try:
        c = spark.read.table(t).count()
        print(f"{t:<30} | {c:>8,} | --OK--")
    except:
        print(f"{t:<30} | {'N/A':>8} | --ERROR-- chưa có")

print("="*55)
print("-> Copy bảng này gửi cho [A] Hà để điền gold_layer_summary.md")


 GOLD LAYER COMPLETE — DAY 3 + DAY 4
Table                          |  Records | Status
--------------------------------------------------
gold_revenue_daily             |      451 | --OK--
gold_provider_performance      |      106 | --OK--
gold_tour_performance          |      433 | --OK--
gold_booking_funnel            |        3 | --OK--
gold_payment_analysis          |        5 | --OK--
gold_payment_success_rate      |        4 | --OK--
gold_review_summary            |      198 | --OK--
gold_rating_distribution       |        3 | --OK--
-> Copy bảng này gửi cho [A] Hà để điền gold_layer_summary.md


### Cell 6 — Gửi data cho [C] PHỤNG (Dashboard)

In [0]:
# ============================================================
#  DATA GỬI CHO [C] KHÁNH — Dashboard Section 4 & 5
# ============================================================

print("="*55)
print(" DATA CHO [C] PHỤNG — Dashboard Sections 4 & 5")
print("="*55)

print("\n▶ SECTION 4: Payment Analysis")
spark.read.table("gold_payment_analysis").show(truncate=False)
spark.read.table("gold_payment_success_rate").show(truncate=False)

print("\n▶ SECTION 5: Review Summary (Top 5)")
spark.read.table("gold_review_summary") \
    .select("title", "avg_rating", "review_count", "five_star") \
    .limit(5) \
    .show(truncate=False)

print("\n-> Rating Distribution (toàn hệ thống)")
spark.read.table("gold_rating_distribution").show(truncate=False)

print("\n-> Gửi screenshot này cho [C] PHỤNG")

 DATA CHO [C] PHỤNG — Dashboard Sections 4 & 5

▶ SECTION 4: Payment Analysis
+--------------+--------------+-----+------------+
|payment_method|status        |count|total_amount|
+--------------+--------------+-----+------------+
|BANK_TRANSFER |SUCCESS       |241  |1.92065E9   |
|CASH          |SUCCESS       |274  |2.50255E9   |
|VNPAY         |SUCCESS       |269  |1.95905E9   |
|VietQR        |AWAITING_ADMIN|6    |5332222.0   |
|VietQR        |SUCCESS       |4    |5140000.0   |
+--------------+--------------+-----+------------+

+--------------+--------------+----------+------------+
|payment_method|total_attempts|successful|success_rate|
+--------------+--------------+----------+------------+
|VNPAY         |269           |269       |100.0       |
|BANK_TRANSFER |241           |241       |100.0       |
|VietQR        |10            |4         |40.0        |
|CASH          |274           |274       |100.0       |
+--------------+--------------+----------+------------+


▶ SECTION 5:

In [0]:
count = spark.read.table("silver_bookings").count()
print(f"Bookings hiện tại: {count}")
print("-> Cần augment" if count < 200 else "→ Đủ data, không cần augment")

Bookings hiện tại: 1326
→ Đủ data, không cần augment
